# VFD Energy Optimization: Pump Affinity Laws

Demonstrates the pump affinity laws — how flow, head, and power scale with pump speed — and why slowing a pump down with a Variable Frequency Drive (VFD) is such an effective energy-saving lever: power scales with the **cube** of speed, so a modest speed reduction yields an outsized power saving.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd

from src.hydraulics.electrical import apply_affinity_laws, speed_ratio_for_target_flow

## Baseline: pump at full speed

Take the four-inch scenario's full-speed operating point as the reference.

In [ ]:
from src.simulation.scenario import run_simulation

baseline = run_simulation(diameter_m=0.1016, flow_rate_m3s=0.0005, length_m=100.0,
                           fittings={'elbow_90_standard': 4, 'gate_valve_open': 1})
Q_ref = baseline.scenario.flow_rate_m3s
H_ref = baseline.total_head_m
P_ref = baseline.pump.shaft_power_W
print(f'Reference point: Q={Q_ref*1000:.3f} L/s, H={H_ref:.4f} m, P={P_ref:.4f} W')

## Sweep speed ratio 50%-100% and observe the cube-law power drop

In [ ]:
speed_ratios = np.linspace(0.5, 1.0, 11)
rows = []
for n in speed_ratios:
    r = apply_affinity_laws(Q_ref, H_ref, P_ref, speed_ratio=n)
    rows.append({'speed_ratio': n, 'flow_m3s': r.flow_m3s, 'head_m': r.head_m, 'power_W': r.power_W})
df = pd.DataFrame(rows)
df['power_pct_of_full'] = df['power_W'] / P_ref * 100
df

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(x=df['speed_ratio']*100, y=df['power_pct_of_full'],
                          mode='lines+markers', name='Power (% of full speed)'))
fig.add_trace(go.Scatter(x=df['speed_ratio']*100, y=df['speed_ratio']*100,
                          mode='lines', name='Speed (% of full)', line=dict(dash='dash')))
fig.update_layout(title='Cube-Law Power Savings vs. Linear Speed Reduction',
                   xaxis_title='Pump speed (% of full speed)',
                   yaxis_title='Power (% of full-speed power)',
                   template='plotly_white')
fig.show()

## The key insight

Notice that at **80% speed**, power drops to roughly **51%** — nearly *half* the power for only a 20% speed reduction. This cube-law relationship is exactly why VFDs are such an effective tool for energy optimization whenever demand fluctuates (e.g. matching pump output to varying downstream water demand instead of running a fixed-speed pump against a throttling valve).

In [ ]:
n_80 = apply_affinity_laws(Q_ref, H_ref, P_ref, speed_ratio=0.8)
savings_pct = (1 - n_80.power_W / P_ref) * 100
print(f'At 80% speed: Q={n_80.flow_m3s*1000:.3f} L/s, H={n_80.head_m:.4f} m, '
      f'P={n_80.power_W:.4f} W')
print(f'Power savings vs. full speed: {savings_pct:.1f}%')

## Solving the other direction: what speed hits a target flow?

If downstream demand drops to 70% of design flow, what speed should the VFD run at, and what's the resulting power draw?

In [ ]:
Q_target = 0.7 * Q_ref
required_ratio = speed_ratio_for_target_flow(Q_ref, Q_target)
result = apply_affinity_laws(Q_ref, H_ref, P_ref, speed_ratio=required_ratio)

print(f'To hit {Q_target*1000:.3f} L/s (70% of design flow):')
print(f'  Required speed: {required_ratio:.1%} of full speed')
print(f'  Resulting head: {result.head_m:.4f} m')
print(f'  Resulting power: {result.power_W:.4f} W '
      f'({result.power_W/P_ref:.1%} of full-speed power)')

## Caveat

The affinity laws assume the pump operates at geometrically similar points on its performance curve as speed changes — accurate for the same pump/system combination over a moderate speed range, but not for comparing different pump models or impeller trims. For precise energy audits, validate against the manufacturer's actual variable-speed performance curves where available.